In [54]:
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pandas as pd
import numpy as np
import os

In [55]:
MLFLOW_TRACKING_URI = "http://mlflow:8080" 

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [56]:
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target

feature_names = diabetes.feature_names
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

In [57]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=55
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [58]:
params = {
    "n_estimators": 125,
    "max_depth": 12,
    "min_samples_split": 6,
    "min_samples_leaf": 3,
    "max_features": 'sqrt',
    "random_state": 55,
    "bootstrap": True,
    "oob_score": True
}

In [59]:
with mlflow.start_run(run_name="diabetes") as run:
    mlflow.log_param("preprocessing", "StandardScaler")
    mlflow.log_param("test_size", 0.25)
    mlflow.log_param("random_state_split", 55)
    
    mlflow.log_params(params)
    
    rf_model = RandomForestRegressor(**params)
    rf_model.fit(X_train_scaled, y_train)
    
    y_train_pred = rf_model.predict(X_train_scaled)
    y_test_pred = rf_model.predict(X_test_scaled)
    
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    train_rmse = np.sqrt(train_mse)
    test_rmse = np.sqrt(test_mse)
    
    cv_scores = cross_val_score(rf_model, X_train_scaled, y_train, cv=6, scoring='r2')
    
    mlflow.log_metrics({
        "train_mse": train_mse,
        "test_mse": test_mse,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "train_r2": train_r2,
        "test_r2": test_r2,
        "train_rmse": train_rmse,
        "test_rmse": test_rmse,
        "cv_mean_r2": cv_scores.mean(),
        "cv_std_r2": cv_scores.std(),
        "oob_score": rf_model.oob_score_
    })
    
    mlflow.sklearn.log_model(
        rf_model,
        "diabetes",
        registered_model_name="diabetes"
    )

2026/03/10 09:35:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/10 09:35:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/10 09:35:34 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'diabetes'.
2026/03/10 09:35:34 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: diabetes, version 1


🏃 View run diabetes at: http://mlflow:8080/#/experiments/0/runs/2985d985278d4717859f5ff0aec28c0a
🧪 View experiment at: http://mlflow:8080/#/experiments/0


Created version '1' of model 'diabetes'.
